# ShopDesk, Module 3 Section 3 Lab 1: Test-Driven Iteration and Interview-Style Prompting

A beginner-friendly notebook on three habits that make code generation reliable: giving concrete
**input/output examples**, working **test-first** (write the tests, watch them fail, then make them
pass), and using an **interview-style** prompt so Claude asks clarifying questions before it writes
code. It also covers when to fix several issues **together** versus **one at a time**. A real
**pytest** loop runs offline; live cells use your **Anthropic API key** with **Sonnet**
(`claude-sonnet-4-6`).

## The real-world scenario

ShopDesk needs a refund-eligibility function. A vague request ("make refunds work") invites the wrong
guess; a few concrete examples and a failing test suite pin down exactly what "correct" means. And when
the spec is fuzzy, the fastest path is for Claude to ask a couple of sharp questions first, not to code
blindly.

The question this lab answers: **how do examples and tests remove ambiguity, and when should Claude ask
before acting?**

## Objectives

- Turn **input/output examples** into a failing test suite, then make it pass (test-driven iteration).
- Use an **interview-style** prompt so Claude asks clarifying questions before coding.
- Choose **combined** vs **sequential** fixes for multiple issues.

## What you'll observe

- pytest goes from red (stub) to green (implementation), driven by the examples.
- A fuzzy spec produces clarifying questions instead of a confident wrong answer.
- Interdependent issues are fixed in one pass; independent issues are fixed separately.

## How to run

Run top to bottom. The pytest loop and the strategy cells run anywhere. The two live cells call Claude,
so paste a real key into **Setup 2/3** and re-run from the top; otherwise they skip.

## 0. Setup

**This cell:** installs the packages. `pytest` runs the test loop; the `anthropic` SDK drives the
live cells.

In [ ]:
# ===== SETUP 1/3 - install the packages =====
%pip install -q anthropic python-dotenv pytest

**This cell:** imports, the model, the `RUN_LIVE` switch, and a client for the live cells.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, a client =====
import os                                       # filesystem paths and env for pytest
import sys                                       # run pytest as a subprocess with this interpreter
import subprocess                               # invoke pytest and capture its output
import re                                       # strip code fences from model output

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cells will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

client = None                                    # the live cells build this only if RUN_LIVE
if RUN_LIVE:                                     # avoid constructing a client with a fake key
    import anthropic                             #   import the SDK
    client = anthropic.Anthropic()              #   reads ANTHROPIC_API_KEY from the environment

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** writes a tiny **ShopDesk package** with a deliberately empty `is_refundable` (a
stub) plus a package marker, so the tests we add next will fail first. This is the starting point of the
test-first loop.

In [ ]:
# ===== SETUP 3/3 - create the package with an unimplemented stub =====
import textwrap                                    # keeps the embedded file bodies readable
REPO = os.path.join(os.getcwd(), "shopdesk_tdd")   # the sample package root

def write(rel, content):                           # small helper: write a file under REPO
    path = os.path.join(REPO, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)

write("shopdesk/__init__.py", "")                  # make shopdesk an importable package
write("shopdesk/refunds.py", textwrap.dedent("""\
    def is_refundable(order):
        # not implemented yet (this is the stub)
        return None
    """))
print("created package at", REPO)

### Examples first, then tests

Concrete **input/output examples** are the cheapest way to remove ambiguity. Before writing code, agree
on what the function returns for specific inputs, including the tricky boundary (exactly 30 days). Those
examples become your tests, and the tests become the definition of done.

---

### Lab objective - let examples and tests drive the code

**What you build:** examples for `is_refundable`, a failing test suite from them, a passing
implementation, an interview gate for fuzzy specs, and a combined-vs-sequential fix strategy.

**Why it helps you build real solutions:** tests turn "looks right" into "is right", and asking first
avoids expensive wrong guesses.

**How you'll see it:** pytest flips from red to green, and the strategy cell routes issues correctly.

**This cell:** the **input/output examples**. The rule: refundable only if the order is delivered
and within 30 days (inclusive). We list the examples explicitly, including the boundary cases, so there
is no room for interpretation.

In [ ]:
# ===== the input/output examples that define "correct" =====
EXAMPLES = [
    ({"status": "delivered", "delivered_days_ago": 5},  True),    # inside the window
    ({"status": "delivered", "delivered_days_ago": 30}, True),    # exactly 30 days -> inclusive
    ({"status": "delivered", "delivered_days_ago": 31}, False),   # one day too late
    ({"status": "shipped",   "delivered_days_ago": 2},  False),   # not delivered yet
]
for order, expected in EXAMPLES:                   # show the contract
    print(f"  {order} -> {expected}")

**This cell:** turns the examples into a **pytest** file. Each example becomes an assertion,
including the 30-day boundary. Writing the tests before the code is the heart of test-driven iteration.

In [ ]:
# ===== generate the test file from the examples =====
test_lines = ["from shopdesk.refunds import is_refundable", "", "def test_examples():"]
for order, expected in EXAMPLES:                   # one assert per example
    test_lines.append(f"    assert is_refundable({order!r}) is {expected}")
write("tests/test_refunds.py", "\n".join(test_lines) + "\n")
print("\n".join(test_lines))

**This cell:** run pytest now, against the stub. Every example fails because `is_refundable`
returns `None`. This is the **red** step: the tests describe the goal and prove we are not there yet.

In [ ]:
# ===== RED: run the tests against the stub =====
def run_pytest():                                  # run pytest in REPO and return its tail
    env = {**os.environ, "PYTHONPATH": REPO}         # let tests import the shopdesk package
    r = subprocess.run([sys.executable, "-m", "pytest", "-q"],
                       cwd=REPO, capture_output=True, text=True, env=env)
    return r.stdout[-700:]

print(run_pytest())

**This cell:** implement `is_refundable` from the examples, then run pytest again. With the rule
written correctly (delivered and within 30 days inclusive), every example passes. This is the **green**
step that closes the loop.

In [ ]:
# ===== GREEN: implement from the examples, then re-run =====
write("shopdesk/refunds.py", textwrap.dedent("""\
    def is_refundable(order):
        # refundable only if delivered and within 30 days (inclusive)
        return order["status"] == "delivered" and order["delivered_days_ago"] <= 30
    """))
print(run_pytest())

**This cell:** the **interview** habit. When a request is fuzzy, coding immediately bakes in a
guess. This gate checks a spec for the aspects that matter and lists the questions to ask first, so
ambiguity is resolved before any code is written.

In [ ]:
# ===== interview gate: what to ask before coding a fuzzy spec =====
ASPECTS = {                                        # aspect -> the question to ask if it is missing
    "boundary":  "Is the 30-day window inclusive or exclusive?",
    "status":    "Which order statuses count as refundable?",
    "rounding":  "How should partial-cent amounts be rounded?",
    "partial":   "Are partial refunds allowed, or full-only?",
}
def interview(spec):                               # spec text -> the clarifying questions it leaves open
    return [q for key, q in ASPECTS.items() if key not in spec.lower()]

fuzzy = "Make refunds work for delivered orders."   # a vague request
for q in interview(fuzzy):                          # the questions Claude should ask first
    print("  ?", q)

**This cell:** the live **interview-style** prompt. We tell Claude to ask clarifying questions
before writing any code, then hand it the fuzzy spec. Instead of guessing, it should come back with
questions, exactly the behavior the gate above encourages.

In [ ]:
# ===== live: ask clarifying questions before coding =====
INTERVIEW_SYS = ("You are a careful engineer. If a request is ambiguous, ask up to 3 clarifying "
                 "questions BEFORE writing any code. Do not write code yet.")

if RUN_LIVE:                                        # needs a real key
    resp = client.messages.create(model=MODEL, max_tokens=400, system=INTERVIEW_SYS,
        messages=[{"role": "user", "content": "Make refunds work for delivered orders."}])
    print("".join(b.text for b in resp.content if b.type == "text").strip()[:600])
else:
    print("[skipped] expected: Claude asks about the window boundary, statuses, and partial refunds.")

**This cell:** live **test-driven implementation**. We reset to the stub, hand Claude only the
tests, and ask for an implementation that passes them. We then run pytest on its output. This is the
loop an agent runs: tests in, code out, verified green.

In [ ]:
# ===== live: let Claude implement from the tests, then verify =====
def strip_fences(text):                             # remove ```python fences if present
    return re.sub(r"^```[a-z]*\n|\n```$", "", text.strip(), flags=re.MULTILINE)

if RUN_LIVE:                                        # needs a real key
    write("shopdesk/refunds.py", "def is_refundable(order):\n    return None\n")   # reset to stub
    tests = open(os.path.join(REPO, "tests/test_refunds.py")).read()                # the spec-as-tests
    resp = client.messages.create(model=MODEL, max_tokens=500,
        messages=[{"role": "user", "content": f"Write shopdesk/refunds.py so these tests pass. Output only Python.\n\n{tests}"}])
    write("shopdesk/refunds.py", strip_fences("".join(b.text for b in resp.content if b.type == "text")))
    print(run_pytest())                             # expect: all tests pass
else:
    print("[skipped] expected: Claude writes is_refundable and pytest reports all passed.")

**This cell:** handling **multiple issues**. Interdependent fixes (one change forces another)
belong in a single pass so they stay consistent; independent fixes are safer done one at a time. This
picks the strategy from whether the issues share a dependency.

In [ ]:
# ===== combined (interdependent) vs sequential (independent) =====
def strategy(issues):                              # issues: list of (name, depends_on_another)
    return "combined" if any(dep for _, dep in issues) else "sequential"

set_a = [("rename REFUND_WINDOW_DAYS", True), ("update callers of the constant", True)]   # linked
set_b = [("fix a docstring typo", False), ("add a missing type hint", False)]             # independent
for name, issues in [("A", set_a), ("B", set_b)]:  # recommend a strategy for each set
    print(f"  set {name}: {strategy(issues)}")

| anti-pattern | what to do instead |
|---|---|
| describe behavior in prose only | give concrete input/output examples |
| write code, then tests (or no tests) | write tests first, watch them fail, then pass |
| guess at a fuzzy spec | ask clarifying questions before coding |
| fix interdependent issues one by one | fix them together so they stay consistent |

**Lesson:** examples and tests convert a fuzzy request into a precise, checkable contract, and the
red-to-green loop keeps the code honest. When the spec is unclear, an interview beats a guess. And match
the fix strategy to the issues: combine the interdependent, separate the independent.

---

## Recap - iterate with tests and questions

| Habit | Move | Payoff |
|---|---|---|
| Examples first | list input/output pairs | removes ambiguity |
| Test-driven | red, then green | code proven correct |
| Interview | ask before coding | avoids wrong guesses |
| Multi-issue | combine or sequence | consistent, safe fixes |

One principle to carry forward: **let the tests define done, and ask before you assume.** To run live,
paste a real key into **Setup 2/3** and re-run from the top. Then try it: add a partial-refund example,
watch the suite go red, and drive it green again. Next lab: running headless in CI with structured JSON.